# 模块一实验：GPU性能分析实战

**目标**：
1. 计算不同操作的算术强度（Arithmetic Intensity）
2. 可视化 Roofline 模型，定位瓶颈
3. 用 PyTorch Profiler 分析 Transformer 推理热点

**环境**：有GPU效果更好，但 M1 Mac / CPU 也能完整运行

```bash
pip install torch matplotlib numpy
```

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import time

# 检测运行环境
if torch.cuda.is_available():
    device = torch.device('cuda')
    gpu_name = torch.cuda.get_device_name(0)
    total_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU: {gpu_name}')
    print(f'显存: {total_mem:.1f} GB')
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    device = torch.device('mps')
    print('Apple Silicon MPS 加速')
else:
    device = torch.device('cpu')
    print('CPU 模式（实验可运行，速度较慢）')

print(f'使用设备: {device}')

---
## Part 1：手动计算算术强度

**算术强度（Arithmetic Intensity）= 计算量（FLOPs）/ 内存访问量（Bytes）**

这个比值决定了操作是 **Compute-Bound**（算力瓶颈）还是 **Memory-Bound**（带宽瓶颈）。

In [ ]:
def arithmetic_intensity_gemm(M, K, N, dtype_bytes=2):
    """矩阵乘法 (M×K) @ (K×N) 的算术强度"""
    flops = 2 * M * K * N          # 每个输出元素需要 K 次乘加
    # 内存访问：读A + 读B + 写C
    bytes_accessed = (M*K + K*N + M*N) * dtype_bytes
    return flops / bytes_accessed

def arithmetic_intensity_attention(seq_len, d_head, num_heads, dtype_bytes=2):
    """Self-Attention 的算术强度（简化估算）"""
    # QK^T: (seq, d_head) @ (d_head, seq) → (seq, seq)
    flops_qk = 2 * seq_len * d_head * seq_len
    # AV: (seq, seq) @ (seq, d_head) → (seq, d_head)
    flops_av = 2 * seq_len * seq_len * d_head
    total_flops = (flops_qk + flops_av) * num_heads
    
    # 内存：Q/K/V 输入 + 注意力矩阵 + 输出
    bytes_qkv = 3 * seq_len * d_head * num_heads * dtype_bytes
    bytes_attn = seq_len * seq_len * num_heads * dtype_bytes  # 注意力矩阵
    bytes_out = seq_len * d_head * num_heads * dtype_bytes
    total_bytes = bytes_qkv + bytes_attn + bytes_out
    return total_flops / total_bytes

# ── 计算各种场景的算术强度 ──
print('=== GEMM（矩阵乘法）算术强度 ===')
cases = [
    ('LLaMA-7B Linear层 (batch=1)',  1,    4096, 4096),
    ('LLaMA-7B Linear层 (batch=32)', 32,   4096, 4096),
    ('LLaMA-7B FFN Up   (batch=1)',  1,    4096, 11008),
    ('LLaMA-7B FFN Up   (batch=32)', 32,   4096, 11008),
]
for name, M, K, N in cases:
    ai = arithmetic_intensity_gemm(M, K, N)
    print(f'  {name:<40} AI = {ai:>8.1f} FLOPs/Byte')

print()
print('=== Self-Attention 算术强度 ===')
# LLaMA-2-7B: 32头，head_dim=128
attn_cases = [
    ('Attention (seq=512,  batch=1)',  512,  128, 32),
    ('Attention (seq=2048, batch=1)', 2048,  128, 32),
    ('Attention (seq=512,  batch=8)',  512,  128, 32),
]
for name, s, d, h in attn_cases:
    ai = arithmetic_intensity_attention(s, d, h)
    print(f'  {name:<40} AI = {ai:>8.1f} FLOPs/Byte')

print()
print('Ridge Point（H100 SXM）≈ 989 TFLOPS / 3350 GB/s ≈ 295 FLOPs/Byte')
print('AI < 295 → Memory-Bound（带宽瓶颈），AI > 295 → Compute-Bound（算力瓶颈）')

---
## Part 2：Roofline 模型可视化

Roofline 图有两条线：
- **水平线**：算力天花板（峰值 TFLOPS）
- **斜线**：带宽天花板（峰值带宽 × 算术强度）

操作点在斜线下方 → Memory-Bound；在水平线下方 → Compute-Bound

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle('Roofline 模型：定位 AI 计算瓶颈', fontsize=13, fontweight='bold')

def plot_roofline(ax, peak_flops_tflops, bandwidth_tbps, title, operations):
    """画 Roofline 图并标注操作点"""
    peak_compute = peak_flops_tflops   # TFLOPS
    peak_bw      = bandwidth_tbps * 1000  # GB/s
    ridge_point  = peak_compute * 1e12 / (peak_bw * 1e9)  # FLOPs/Byte

    ai_range = np.logspace(-1, 4, 500)
    perf     = np.minimum(peak_bw * ai_range / 1000, peak_compute)  # TFLOPS

    ax.loglog(ai_range, perf, 'b-', linewidth=2.5, label='Roofline')
    ax.fill_between(ai_range, 0, perf, alpha=0.08, color='blue')
    ax.axvline(x=ridge_point, color='gray', linestyle='--', alpha=0.6, linewidth=1)

    ax.text(ridge_point * 0.25, peak_compute * 0.15,
            'Memory\nBound', color='red', fontsize=10, fontweight='bold', ha='center')
    ax.text(ridge_point * 3.5, peak_compute * 0.15,
            'Compute\nBound', color='green', fontsize=10, fontweight='bold', ha='center')

    colors = ['#e74c3c','#e67e22','#2ecc71','#3498db','#9b59b6','#1abc9c']
    for idx, (name, ai_val, marker) in enumerate(operations):
        perf_val = min(peak_bw * ai_val / 1000, peak_compute)
        c = colors[idx % len(colors)]
        ax.scatter(ai_val, perf_val, s=120, color=c, zorder=5, marker=marker)
        ax.annotate(name, (ai_val, perf_val),
                    xytext=(ai_val * 1.4, perf_val * 0.6),
                    fontsize=8, color=c,
                    arrowprops=dict(arrowstyle='->', color=c, lw=1))

    ax.set_xlabel('算术强度 (FLOPs/Byte)', fontsize=11)
    ax.set_ylabel('实际性能 (TFLOPS)', fontsize=11)
    ax.set_title(f'{title}\n(Ridge Point ≈ {ridge_point:.0f} FLOPs/Byte)', fontsize=11)
    ax.grid(True, alpha=0.3)
    ax.set_xlim(0.1, 5000)
    ax.set_ylim(0.01, peak_compute * 2)
    return ridge_point

# H100 SXM 规格
h100_ops = [
    ('Linear(batch=1)',   0.5,  'v'),
    ('Linear(batch=32)',  16,   's'),
    ('Attn(seq=512,b=1)', 2.1,  'o'),
    ('Attn(seq=2048,b=1)',3.8,  'o'),
    ('FFN(batch=32)',     22,   '^'),
    ('Dense GEMM(train)', 450,  '*'),
]
rp1 = plot_roofline(axes[0], 989, 3.35, 'H100 SXM (FP16)', h100_ops)

# A100 SXM 规格
a100_ops = [
    ('Linear(batch=1)',   0.5,  'v'),
    ('Linear(batch=32)',  16,   's'),
    ('Attn(seq=512,b=1)', 2.1,  'o'),
    ('FFN(batch=32)',     22,   '^'),
]
rp2 = plot_roofline(axes[1], 312, 2.0, 'A100 SXM (FP16)', a100_ops)

plt.tight_layout()
plt.savefig('roofline_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\nH100 Ridge Point: {rp1:.0f} FLOPs/Byte')
print(f'A100 Ridge Point: {rp2:.0f} FLOPs/Byte')
print('\n结论：batch=1 的 Linear 层 AI ≈ 0.5，远低于 Ridge Point → 严重 Memory-Bound')
print('      增大 batch_size 可以提升 AI，朝 Compute-Bound 移动')

---
## Part 3：PyTorch Profiler 实战

用 Profiler 找出 Transformer 推理中最耗时的操作。

In [ ]:
class TransformerBlock(nn.Module):
    """简化版 Transformer Block（用于实验）"""
    def __init__(self, d_model=512, num_heads=8, d_ff=2048):
        super().__init__()
        self.attn  = nn.MultiheadAttention(d_model, num_heads, batch_first=True)
        self.ffn   = nn.Sequential(
            nn.Linear(d_model, d_ff), nn.GELU(), nn.Linear(d_ff, d_model)
        )
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x):
        a, _ = self.attn(x, x, x)
        x = self.norm1(x + a)
        x = self.norm2(x + self.ffn(x))
        return x

# 构建小模型（6层，适合 CPU/MPS 运行）
model = nn.Sequential(*[TransformerBlock() for _ in range(6)]).to(device)
model.eval()

batch_size, seq_len, d_model = 4, 256, 512
x = torch.randn(batch_size, seq_len, d_model, device=device)

# 预热
with torch.no_grad():
    for _ in range(3):
        _ = model(x)

print(f'模型参数量: {sum(p.numel() for p in model.parameters())/1e6:.1f}M')
print(f'输入形状: batch={batch_size}, seq={seq_len}, d_model={d_model}')

In [ ]:
from torch.profiler import profile, record_function, ProfilerActivity

activities = [ProfilerActivity.CPU]
if torch.cuda.is_available():
    activities.append(ProfilerActivity.CUDA)

with profile(
    activities=activities,
    record_shapes=True,
    with_flops=True,
) as prof:
    with torch.no_grad():
        with record_function('transformer_inference'):
            out = model(x)

# 打印 Top-15 耗时操作
print('=== Top-15 耗时操作（按 CPU 时间排序）===')
print(prof.key_averages().table(
    sort_by='cpu_time_total', row_limit=15
))

In [ ]:
# 可视化：各操作类型耗时占比
events = prof.key_averages()

op_times = {}
for e in events:
    if e.cpu_time_total > 0 and not e.key.startswith('aten::_'):
        name = e.key.replace('aten::', '')
        if name in ['mm', 'bmm', 'addmm', 'linear']:
            op_times['GEMM/Linear'] = op_times.get('GEMM/Linear', 0) + e.cpu_time_total
        elif 'norm' in name.lower():
            op_times['LayerNorm'] = op_times.get('LayerNorm', 0) + e.cpu_time_total
        elif name in ['scaled_dot_product_attention', 'softmax']:
            op_times['Attention'] = op_times.get('Attention', 0) + e.cpu_time_total
        elif name in ['gelu', 'relu']:
            op_times['Activation'] = op_times.get('Activation', 0) + e.cpu_time_total
        elif e.cpu_time_total > 1000:  # 超过1ms的其他操作
            op_times['Other'] = op_times.get('Other', 0) + e.cpu_time_total

if op_times:
    fig, ax = plt.subplots(figsize=(8, 5))
    labels = list(op_times.keys())
    values = [op_times[k] / 1000 for k in labels]  # 转换为 ms
    colors = ['#3498db', '#e74c3c', '#2ecc71', '#f39c12', '#9b59b6']
    bars = ax.bar(labels, values, color=colors[:len(labels)], alpha=0.85, edgecolor='black')
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                f'{val:.2f}ms', ha='center', fontsize=10)
    ax.set_ylabel('CPU 耗时 (ms)')
    ax.set_title('Transformer 各操作类型耗时分布')
    ax.grid(True, alpha=0.3, axis='y')
    plt.tight_layout()
    plt.savefig('profiler_breakdown.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('\nGEMM/Linear 通常占 60-80%——这是 Transformer 最主要的计算瓶颈')
else:
    print('Profile 数据解析中，请查看上方 table 输出')

---
## Part 4：Batch Size 对算术强度的影响

验证「增大 batch size 可以让 Linear 层从 Memory-Bound 走向 Compute-Bound」的直觉。

In [ ]:
# LLaMA-2-7B Linear层 (d_model=4096, d_ff=11008)
batch_sizes  = [1, 2, 4, 8, 16, 32, 64, 128]
K, N         = 4096, 4096

ai_values    = [arithmetic_intensity_gemm(bs, K, N) for bs in batch_sizes]
h100_ridge   = 989e12 / (3.35e12)   # ≈ 295 FLOPs/Byte

fig, ax = plt.subplots(figsize=(10, 5))
ax.semilogx(batch_sizes, ai_values, 'b-o', linewidth=2, markersize=8, label='Linear层 AI值')
ax.axhline(y=h100_ridge, color='red', linestyle='--', linewidth=2,
           label=f'H100 Ridge Point ({h100_ridge:.0f} FLOPs/Byte)')

# 标注 Memory-Bound / Compute-Bound 区域
ax.fill_between(batch_sizes, 0, h100_ridge,
                alpha=0.1, color='red', label='Memory-Bound 区域')
ax.fill_between(batch_sizes, h100_ridge, max(ai_values) * 1.2,
                alpha=0.1, color='green', label='Compute-Bound 区域')

for bs, ai in zip(batch_sizes, ai_values):
    ax.annotate(f'{ai:.0f}', (bs, ai), textcoords='offset points',
                xytext=(0, 8), ha='center', fontsize=9)

ax.set_xlabel('Batch Size', fontsize=12)
ax.set_ylabel('算术强度 (FLOPs/Byte)', fontsize=12)
ax.set_title('增大 Batch Size → Linear层算术强度提升\n(LLaMA-2-7B 第一个 Linear层, K=N=4096, FP16)', fontsize=11)
ax.set_xticks(batch_sizes)
ax.set_xticklabels(batch_sizes)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('batch_size_vs_ai.png', dpi=150, bbox_inches='tight')
plt.show()

# 找到进入 Compute-Bound 的最小 batch size
for bs, ai in zip(batch_sizes, ai_values):
    if ai >= h100_ridge:
        print(f'\nBatch Size ≥ {bs} 时，Linear层进入 Compute-Bound 区域 (AI={ai:.0f})')
        break
else:
    print(f'\n在测试的 batch_sizes 范围内，Linear层仍处于 Memory-Bound 区域')
    print(f'最大 AI = {max(ai_values):.0f}，H100 Ridge = {h100_ridge:.0f}')

---
## 思考题

完成实验后，请思考并记录你的答案：

**Q1**：从 Part 1 的计算结果来看，`Attention (seq=512, batch=1)` 的算术强度是多少？它属于 Memory-Bound 还是 Compute-Bound？增大 seq_len 会让 AI 升高还是降低？为什么？

*(你的答案)*：

---

**Q2**：Flash Attention 的核心改进是减少 HBM 读写次数。在 Roofline 图上，这个改进让 Attention 的操作点如何移动？（提示：FLOPs 不变，Bytes 减少，AI 如何变化？）

*(你的答案)*：

---

**Q3**：你的运行环境是什么（GPU型号 / MPS / CPU）？如果是 GPU，它的峰值算力和 HBM 带宽分别是多少？Ridge Point 在哪里？（可查 NVIDIA 官方规格表）

*(你的答案)*：

---

**Q4**：从 Part 3 的 Profiler 结果来看，哪类操作占用了最多 CPU 时间？这和你对 Transformer 的预期一致吗？

*(你的答案)*：